<a href="https://colab.research.google.com/github/awinarko-hue/Project_hue/blob/main/OnWork/GraphRAG_tesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## S0 — Setup Lingkungan

In [ ]:
!pip install openai python-dotenv

In [ ]:
# Instalasi dependensi (sekali per sesi Colab)
!pip install -q sentence-transformers faiss-cpu rank_bm25 networkx pyvis google-genai

import os, re, glob, json, time, pickle
import numpy as np
import pandas as pd
import networkx as nx
pd.set_option('display.max_colwidth', 120)
print("Setup selesai.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 101.5 MB/s eta 0:00:00
Setup selesai.


In [ ]:
MOUNT_DRIVE = True

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR   = '/content/drive/MyDrive/Tesis_SDPPI/Project 7/Aplikasi_Skema/GraphRAG_Komdigi'   # sesuaikan
else:
    BASE_DIR   = '/content/GraphRAG_Komdigi'

KORPUS_DIR   = os.path.join(BASE_DIR, '/content/drive/MyDrive/Tesis_SDPPI/Project 7/Aplikasi_Skema/Korpus Komdigi')
IFACT_DIR = os.path.join(BASE_DIR, 'artifacts')   # indeks FAISS, graf, embedding
os.makedirs(KORPUS_DIR, exist_ok=True)
os.makedirs(IFACT_DIR, exist_ok=True) # Changed ARTIFACT_DIR to IFACT_DIR

# Model embedding multilingual (konsisten dengan pipeline FAISS Anda sebelumnya).
# Alternatif lebih akurat namun lebih berat: 'intfloat/multilingual-e5-base'
EMBED_MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'

print("Korpus dicari di:", KORPUS_DIR)
print("File terdeteksi :", sorted(os.path.basename(f) for f in glob.glob(os.path.join(KORPUS_DIR, 'korpus*.csv'))))

Mounted at /content/drive
Korpus dicari di: /content/drive/MyDrive/Tesis_SDPPI/Project 7/Aplikasi_Skema/Korpus Komdigi
File terdeteksi : ['korpusPM172018.csv', 'korpusPM42025.csv', 'korpusPM52021.csv', 'korpusPM82024.csv', 'korpusPM82026.csv', 'korpusPM92023.csv', 'korpusPM_2_2023.csv', 'korpusPM_2_2025.csv', 'korpusPP282025.csv', 'korpusPP462021.csv', 'korpusPP492005.csv', 'korpusPP512002.csv', 'korpusPP522000.csv', 'korpusPP532000.csv', 'korpusPerppu22022.csv', 'korpusPerpres_174_2024.csv', 'korpusUU122011.csv', 'korpusUU322002.csv', 'korpusUU361999.csv', 'korpusUU382009.csv', 'korpusUU62023.csv']


## S1 — Tahap 1: Persiapan Data
Memuat seluruh `korpus*.csv`, membersihkan teks, menstandarkan metadata, membentuk
`chunk_id` unik per satuan hukum, dan memvalidasi referensi antar-pasal.

In [ ]:
SCHEMA = ['doc_id','nama_dokumen','jenis','tahun','nomor','bab','nama_bab','bagian',
          'paragraf','pasal','ayat','huruf','angka','tipe','isi','referensi','kata_kunci','entitas']

def load_korpus(korpus_dir: str) -> pd.DataFrame:
    frames, laporan = [], []
    for path in sorted(glob.glob(os.path.join(korpus_dir, 'korpus*.csv'))):
        df = pd.read_csv(path)
        hilang = [c for c in SCHEMA if c not in df.columns]
        if hilang:
            print(f"[LEWATI] {os.path.basename(path)} — kolom hilang: {hilang}")
            continue
        df = df[SCHEMA].copy()
        df['sumber_file'] = os.path.basename(path)
        frames.append(df)
        laporan.append((os.path.basename(path), len(df), df.doc_id.iloc[0]))
    korpus = pd.concat(frames, ignore_index=True)
    print(pd.DataFrame(laporan, columns=['file','baris','doc_id']).to_string(index=False))
    return korpus

def bersihkan(korpus: pd.DataFrame) -> pd.DataFrame:
    df = korpus.copy()
    # 1. Normalisasi teks
    df['isi'] = (df['isi'].astype(str)
                   .str.replace(r'\s+', ' ', regex=True)
                   .str.strip())
    df = df[df['isi'].str.len() > 0].copy()
    # 2. Standarisasi metadata
    df['jenis'] = df['jenis'].astype(str).str.strip()
    df['tahun'] = pd.to_numeric(df['tahun'], errors='coerce').astype('Int64')
    for c in ['pasal','ayat','angka']:
        df[c] = (df[c].astype('string').str.strip()
                       .str.replace(r'\.0$', '', regex=True))   # '3.0' → '3'
    df['huruf'] = df['huruf'].astype('string').str.strip().str.lower()
    # 3. Buang duplikat persis
    n0 = len(df)
    df = df.drop_duplicates(subset=['doc_id','pasal','ayat','huruf','angka','isi'])
    if n0 - len(df): print(f"Duplikat dibuang: {n0-len(df)}")
    # 4. chunk_id unik & hirarkis, mis. UU_6_2023__P154A__a1__h-g__n4
    def buat_id(r):
        parts = [str(r['doc_id']), f"P{r['pasal']}" if pd.notna(r['pasal']) else "P-"]
        if pd.notna(r['ayat']):  parts.append(f"a{r['ayat']}")
        if pd.notna(r['huruf']): parts.append(f"h{r['huruf']}")
        if pd.notna(r['angka']): parts.append(f"n{r['angka']}")
        return "__".join(parts)
    df['chunk_id'] = df.apply(buat_id, axis=1)
    # jika masih ada bentrok id (unit sama terpecah beberapa baris), beri sufiks urut
    dup = df.duplicated('chunk_id', keep=False)
    df.loc[dup, 'chunk_id'] = (df.loc[dup, 'chunk_id'] + '__' +
                               df.loc[dup].groupby('chunk_id').cumcount().astype(str))
    df = df.reset_index(drop=True)
    return df

korpus = bersihkan(load_korpus(KORPUS_DIR))
print(f"\nTotal satuan hukum: {len(korpus)} | Dokumen: {korpus.doc_id.nunique()}")
korpus.head(3)

                      file  baris           doc_id
        korpusPM172018.csv    630   Permen_17_2018
         korpusPM42025.csv      5    Permen_4_2025
         korpusPM52021.csv   1463    Permen_5_2021
         korpusPM82024.csv    246    Permen_8_2024
         korpusPM82026.csv    418    Permen_8_2026
         korpusPM92023.csv    436    Permen_9_2023
       korpusPM_2_2023.csv    323    Permen_2_2023
       korpusPM_2_2025.csv    163    Permen_2_2025
        korpusPP282025.csv   2560       PP_28_2025
        korpusPP462021.csv    611       PP_46_2021
        korpusPP492005.csv     21       PP_49_2005
        korpusPP512002.csv    318       PP_51_2005
        korpusPP522000.csv    417       PP_52_2000
        korpusPP532000.csv    177       PP_53_2000
     korpusPerppu22022.csv   6607    Perppu_2_2022
korpusPerpres_174_2024.csv    123 Perpres_174_2024
        korpusUU122011.csv    981       UU_12_2011
        korpusUU322002.csv    399       UU_32_2002
        korpusUU361999.csv    2

,doc_id,nama_dokumen,jenis,tahun,nomor,bab,nama_bab,bagian,paragraf,pasal,ayat,huruf,angka,tipe,isi,referensi,kata_kunci,entitas,sumber_file,chunk_id
0,Permen_17_2018,Permen Nomor 17 Tahun 2018,Permen,2018,17,BAB I,Ketentuan Umum,NaN,NaN,1,<NA>,<NA>,<NA>,Definisi,Dalam Peraturan Menteri ini yang dimaksud dengan:,Pasal 1,menteri,Menteri,korpusPM172018.csv,Permen_17_2018__P1
1,Permen_17_2018,Permen Nomor 17 Tahun 2018,Permen,2018,17,BAB I,Ketentuan Umum,NaN,NaN,1,<NA>,<NA>,1,Ketentuan,Komunikasi Radio adalah telekomunikasi dengan mempergunakan gelombang radio.,Pasal 1 angka 1,"radio, komunikasi, telekomunikasi, mempergunakan, gelombang",NaN,korpusPM172018.csv,Permen_17_2018__P1__n1
2,Permen_17_2018,Permen Nomor 17 Tahun 2018,Permen,2018,17,BAB I,Ketentuan Umum,NaN,NaN,1,<NA>,<NA>,2,Ketentuan,"Kegiatan Amatir Radio adalah Komunikasi Radio mengenai ilmu pengetahuan, penyelidikan teknis dan informasi yang berk...",Pasal 1 angka 2,"radio, kegiatan, amatir, komunikasi, mengenai",NaN,korpusPM172018.csv,Permen_17_2018__P1__n2


In [ ]:
POLA_INTERNAL  = re.compile(r'Pasal\s+(\d+[A-Z]*)(?:\s+ayat\s*\((\d+)\))?', re.I)
POLA_EKSTERNAL = re.compile(
    r'(Undang-Undang|Peraturan\s+Pemerintah(?:\s+Pengganti\s+Undang-Undang)?|'
    r'Peraturan\s+Presiden|Peraturan\s+Menteri)\s+Nomor\s+(\d+)\s+Tahun\s+(\d{4})', re.I)

JENIS_MAP = {'undang-undang':'UU', 'peraturan pemerintah':'PP',
             'peraturan pemerintah pengganti undang-undang':'Perppu',
             'peraturan presiden':'Perpres', 'peraturan menteri':'Permen'}

def ekstrak_rujukan(df: pd.DataFrame):
    internal, eksternal = [], []
    doc_lookup = {}   # (jenis, nomor, tahun) -> doc_id
    for _, r in df.drop_duplicates('doc_id').iterrows():
        doc_lookup[(str(r['jenis']).lower(), str(r['nomor']), str(r['tahun']))] = r['doc_id']
    pasal_ada = set(zip(df['doc_id'], df['pasal'].astype(str)))
    for _, r in df.iterrows():
        teks = r['isi']
        for m in POLA_INTERNAL.finditer(teks):
            p_tuju = m.group(1)
            if str(p_tuju) != str(r['pasal']):     # abaikan rujukan ke diri sendiri
                internal.append({'chunk_id': r['chunk_id'], 'doc_id': r['doc_id'],
                                 'pasal_asal': r['pasal'], 'pasal_tuju': p_tuju,
                                 'ayat_tuju': m.group(2),
                                 'valid': (r['doc_id'], p_tuju) in pasal_ada})
        for m in POLA_EKSTERNAL.finditer(teks):
            jenis_key = re.sub(r'\s+', ' ', m.group(1).lower())
            jenis = JENIS_MAP.get(jenis_key, m.group(1))
            key = (jenis.lower(), m.group(2), m.group(3))
            tuju = None
            for (j, n, t), did in doc_lookup.items():
                if n == m.group(2) and t == m.group(3):
                    tuju = did; break
            eksternal.append({'chunk_id': r['chunk_id'], 'doc_id': r['doc_id'],
                              'jenis_tuju': jenis, 'nomor': m.group(2), 'tahun': m.group(3),
                              'doc_tuju': tuju, 'dalam_korpus': tuju is not None})
    return pd.DataFrame(internal), pd.DataFrame(eksternal)

ref_internal, ref_eksternal = ekstrak_rujukan(korpus)

print("=== Rujukan internal antar-pasal ===")
print(f"Total: {len(ref_internal)} | valid (pasal tujuan ada): {ref_internal['valid'].sum()}"
      f" | tak ditemukan: {(~ref_internal['valid']).sum()}")
print("\n=== Rujukan antar-dokumen ===")
print(f"Total sebutan: {len(ref_eksternal)} | menuju dokumen dalam korpus: {ref_eksternal['dalam_korpus'].sum()}")
print("\nRegulasi luar korpus yang paling sering dirujuk (kandidat perluasan korpus):")
print(ref_eksternal[~ref_eksternal.dalam_korpus]
      .groupby(['jenis_tuju','nomor','tahun']).size().sort_values(ascending=False).head(10))

=== Rujukan internal antar-pasal ===
Total: 6744 | valid (pasal tujuan ada): 6426 | tak ditemukan: 318

=== Rujukan antar-dokumen ===
Total sebutan: 221 | menuju dokumen dalam korpus: 41

Regulasi luar korpus yang paling sering dirujuk (kandidat perluasan korpus):
jenis_tuju  nomor  tahun
UU          3      1989     7
            11     2020     7
            36     2000     6
            7      1992     6
            25     1992     4
            21     2008     4
Perppu      1      2000     4
UU          2      1981     4
            5      1997     4
            41     1999     4
dtype: int64


In [ ]:
korpus['nomor'] = korpus['nomor'].apply(str)
korpus.to_parquet(os.path.join(IFACT_DIR, 'korpus_bersih.parquet'))
ref_internal.to_csv(os.path.join(IFACT_DIR, 'ref_internal.csv'), index=False)
ref_eksternal.to_csv(os.path.join(IFACT_DIR, 'ref_eksternal.csv'), index=False)
print("Artefak Tahap 1 tersimpan di", IFACT_DIR)

Artefak Tahap 1 tersimpan di /content/drive/MyDrive/Tesis_SDPPI/Project 7/Aplikasi_Skema/GraphRAG_Komdigi/artifacts


## S2 — Tahap 2: Baseline RAG
Chunking sudah alami mengikuti struktur hukum (1 baris = 1 satuan pasal/ayat/huruf/angka).
Setiap chunk diberi *header* konteks (nama dokumen + posisi) agar embedding lebih diskriminatif.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

def teks_chunk(r) -> str:
    lokasi = f"Pasal {r['pasal']}"
    if pd.notna(r['ayat']):  lokasi += f" ayat ({r['ayat']})"
    if pd.notna(r['huruf']): lokasi += f" huruf {r['huruf']}"
    if pd.notna(r['angka']): lokasi += f" angka {r['angka']}"
    return f"{r['nama_dokumen']} — {r['nama_bab']} — {lokasi} [{r['tipe']}]: {r['isi']}"

korpus['teks_embed'] = korpus.apply(teks_chunk, axis=1)

embedder = SentenceTransformer(EMBED_MODEL_NAME)

EMB_PATH = os.path.join(IFACT_DIR, 'embeddings.npy')
IDX_PATH = os.path.join(IFACT_DIR, 'faiss.index')

def bangun_indeks(force=False):
    if (not force) and os.path.exists(EMB_PATH) and os.path.exists(IDX_PATH):
        emb = np.load(EMB_PATH)
        if len(emb) == len(korpus):           # korpus belum berubah → pakai cache
            return emb, faiss.read_index(IDX_PATH)
        print("Ukuran korpus berubah — indeks dibangun ulang.")
    emb = embedder.encode(korpus['teks_embed'].tolist(), batch_size=64,
                          show_progress_bar=True, normalize_embeddings=True)
    emb = np.asarray(emb, dtype='float32')
    index = faiss.IndexFlatIP(emb.shape[1])   # inner product = cosine (sudah dinormalisasi)
    index.add(emb)
    np.save(EMB_PATH, emb); faiss.write_index(index, IDX_PATH)
    return emb, index

embeddings, faiss_index = bangun_indeks()
print("Indeks FAISS:", faiss_index.ntotal, "vektor, dim", embeddings.shape[1])

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indeks FAISS: 21073 vektor, dim 384


In [ ]:
def retrieve_dense(query: str, k: int = 10) -> pd.DataFrame:
    qv = embedder.encode([query], normalize_embeddings=True).astype('float32')
    skor, idx = faiss_index.search(qv, k)
    hasil = korpus.iloc[idx[0]].copy()
    hasil['skor'] = skor[0]
    return hasil[['chunk_id','nama_dokumen','pasal','ayat','tipe','isi','skor']]

retrieve_dense("sanksi bagi penggunaan frekuensi radio tanpa izin", k=5)

,chunk_id,nama_dokumen,pasal,ayat,tipe,isi,skor
740,Permen_9_2023__P63__a2,Permen Nomor 9 Tahun 2023,63,2,Kewajiban,Pelanggaran pemenuhan kewajiban penggunaan Spektrum Frekuensi Radio berdasarkan ISR berupa penggunaan Spektrum Freku...,0.850081
2639,PP_28_2025__P493__a1,PP Nomor 28 Tahun 2025,493,1,Sanksi,Setiap pemegang lzin pita frekuensi radio yang berdasarkan hasil Pengawasan ditemukan melakukan pengalihan hak pengg...,0.849243
257,Permen_8_2026__P9__ha__24,Permen Nomor 8 Tahun 2026,9,<NA>,Larangan,dilarang menimbulkan gangguan yang merugikan (harmful interference) terhadap penggunaan lain dalam Dinas Tetap pada ...,0.843998
269,Permen_8_2026__P9__hb__41,Permen Nomor 8 Tahun 2026,9,<NA>,Larangan,dilarang menyebabkan gangguan yang merugikan (harmful interference) terhadap penggunaan Pita Frekuensi Radio lain ya...,0.835601
772,Permen_9_2023__P72__ha,Permen Nomor 9 Tahun 2023,72,<NA>,Hak,Peraturan Menteri Komunikasi dan Informatika Nomor 19/PER.KOMINFO/10/2005 tentang Petunjuk Pelaksanaan Tarif atas Pe...,0.833215


In [ ]:
# ── LLM: Gemini (gratis di Colab) — bisa diganti API lain ─────────
import google.generativeai as genai
from google.colab import userdata

USE_LLM = True
try:
    genai.configure(api_key=userdata.get('GEMINI_API_KEY'))   # simpan di Secrets Colab (ikon kunci)
    llm = genai.GenerativeModel('gemini-2.5-flash')
except Exception as e:
    USE_LLM = False
    print("LLM nonaktif (API key belum diset). Jawaban akan berupa konteks mentah.\n", e)

PROMPT_RAG = """Anda adalah asisten hukum untuk regulasi Kementerian Komunikasi dan Digital (Komdigi).
Jawab pertanyaan HANYA berdasarkan konteks regulasi berikut. Selalu sebutkan dasar hukumnya
(nama peraturan, pasal, ayat). Jika konteks tidak memadai, katakan tidak ditemukan.

KONTEKS:
{konteks}

PERTANYAAN: {pertanyaan}

JAWABAN (bahasa Indonesia formal, dengan sitasi pasal):"""

def rakit_konteks(df_hasil: pd.DataFrame) -> str:
    baris = []
    for _, r in df_hasil.iterrows():
        lok = f"Pasal {r['pasal']}" + (f" ayat ({r['ayat']})" if pd.notna(r.get('ayat')) else "")
        baris.append(f"[{r.get('nama_dokumen', r['chunk_id'])} — {lok}] {r['isi']}")
    return "\n\n".join(baris)

def jawab(pertanyaan: str, retriever, k: int = 8):
    t0 = time.time()
    hasil = retriever(pertanyaan, k)
    t_ret = time.time() - t0
    konteks = rakit_konteks(hasil)
    if USE_LLM:
        t0 = time.time()
        out = llm.generate_content(PROMPT_RAG.format(konteks=konteks, pertanyaan=pertanyaan)).text
        t_llm = time.time() - t0
    else:
        out, t_llm = "(LLM nonaktif — konteks teratas:)\n" + konteks[:1500], 0.0
    return {'jawaban': out, 'konteks': hasil, 'waktu_retrieval': t_ret, 'waktu_llm': t_llm}

res = jawab("Apa kewajiban penyelenggara pos dalam menjaga kerahasiaan kiriman?", retrieve_dense)
print(res['jawaban'])
print(f"\n[retrieval {res['waktu_retrieval']:.2f}s | LLM {res['waktu_llm']:.2f}s]")

LLM nonaktif (API key belum diset). Jawaban akan berupa konteks mentah.
 Secret GEMINI_API_KEY does not exist.
(LLM nonaktif — konteks teratas:)
[UU Nomor 12 Tahun 2011 — Pasal 30] Penyelenggara pos wajib menjaga kerahasiaan, keamanan, dan keselamatan kiriman.

[UU Nomor 38 Tahun 2009 — Pasal 27 ayat (2)] Pengguna layanan pos berhak atas jaminan kerahasiaan, keamanan, dan keselamatan kiriman.

[UU Nomor 38 Tahun 2009 — Pasal 29 ayat (1)] Penyelenggara Pos berhak mendapatkan informasi yang benar dari pengguna layanan pos tentang kiriman yang dinyatakan pada dokumen pengiriman.

[UU Nomor 38 Tahun 2009 — Pasal 29 ayat (2)] Penyelenggara Pos berhak membuka dan/atau memeriksa kiriman di hadapan pengguna layanan pos untuk mencocokkan kebenaran informasi kiriman sebagaimana dimaksud pada ayat (1).

[UU Nomor 38 Tahun 2009 — Pasal 14 ayat (1)] Penyelenggara Pos wajib menyediakan Jaringan Pos sesuai dengan izin penyelenggaraannya.

[UU Nomor 38 Tahun 2009 — Pasal 30] Penyelenggara Pos wajib me

In [ ]:
# — LLM: GPT-OSS 120B via Groq (Compatible OpenAI, GRATIS) —
# Catatan: llama-3.3-70b-versatile sudah di-deprecate Groq per 16 Agustus 2026.
# Model ini adalah pengganti resmi yang direkomendasikan Groq.
import os
import time
import pandas as pd
from openai import OpenAI
from google.colab import userdata

USE_LLM = True
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    client = OpenAI(
        api_key=os.getenv("GROQ_API_KEY"),
        base_url="https://api.groq.com/openai/v1"
    )
    LLM_MODEL = "openai/gpt-oss-120b"  # pengganti resmi llama-3.3-70b-versatile
except Exception as e:
    USE_LLM = False
    print("LLM nonaktif (API key belum diset). Jawaban akan berupa konteks mentah.\n", e)

PROMPT_RAG = """Anda adalah asisten hukum untuk regulasi Kementerian Komunikasi dan Digital (Komdigi).
Jawab pertanyaan HANYA berdasarkan konteks regulasi berikut. Selalu sebutkan dasar hukumnya
(nama peraturan, pasal, ayat). Jika konteks tidak memadai, katakan tidak ditemukan.
KONTEKS:
{konteks}
PERTANYAAN: {pertanyaan}
JAWABAN (bahasa Indonesia formal, dengan sitasi pasal):"""

def rakit_konteks(df_hasil: pd.DataFrame) -> str:
    baris = []
    for _, r in df_hasil.iterrows():
        lok = f"Pasal {r['pasal']}" + (f" ayat ({r['ayat']})" if pd.notna(r.get('ayat')) else "")
        baris.append(f"[{r.get('nama_dokumen', r['chunk_id'])} — {lok}] {r['isi']}")
    return "\n\n".join(baris)

def jawab(pertanyaan: str, retriever, k: int = 8):
    t0 = time.time()
    hasil = retriever(pertanyaan, k)  # hasil harusnya DataFrame
    t_ret = time.time() - t0
    konteks = rakit_konteks(hasil)
    if USE_LLM:
        t0 = time.time()
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "Anda adalah asisten hukum yang akurat dan tidak mengarang."},
                {
                    "role": "user",
                    "content": PROMPT_RAG.format(konteks=konteks, pertanyaan=pertanyaan)
                }
            ],
            temperature=0.1,
        )
        out = response.choices[0].message.content
        t_llm = time.time() - t0
    else:
        out, t_llm = "(LLM nonaktif — konteks teratas:)\n" + konteks[:1500], 0.0
    return {'jawaban': out, 'konteks': hasil, 'waktu_retrieval': t_ret, 'waktu_llm': t_llm}

res = jawab("Apa kewajiban penyelenggara penyiaran?", retrieve_dense)
print(res['jawaban'])
print(f"\n[retrieval {res['waktu_retrieval']:.2f}s | LLM {res['waktu_llm']:.2f}s]")

**Kewajiban penyelenggara penyiaran (Lembaga Penyiaran Komunitas) menurut PP Nomor 51 Tahun 2005**  

1. **Mendapatkan izin penyelenggaraan penyiaran**  
   - *Pasal 8 ayat (1)*: “Sebelum menyelenggarakan kegiatan, Lembaga Penyiaran Komunitas wajib memperoleh izin penyelenggaraan penyiaran.”

2. **Menanggung tanggung jawab atas seluruh penyelenggaraan penyiaran**  
   - *Pasal 32*: “Pemimpin utama bertanggung jawab atas seluruh penyelenggaraan penyiaran, baik ke dalam maupun ke luar lembaga.”

3. **Merencanakan, melaksanakan, dan mengawasi operasional teknik penyiaran**  
   - *Pasal 31 ayat (4)*: “Penanggung jawab teknik bertugas merencanakan, melaksanakan, dan mengawasi operasional teknik penyiaran.”

4. **Memenuhi standar persyaratan alat dan perangkat penyiaran**  
   - *Pasal 37 ayat (1)*: “Setiap alat dan perangkat penyiaran yang digunakan atau dioperasikan untuk keperluan penyelenggaraan penyiaran wajib memiliki standar persyaratan …”

5. **Menanggapi pengaduan terkait kode etik

## S3 — Tahap 3: Hybrid Retrieval
Dense (FAISS) + sparse (BM25) digabung dengan **Reciprocal Rank Fusion (RRF)**,
ditambah filter metadata: jenis regulasi, tahun, bab, pasal, tipe norma, entitas, kata kunci.

In [ ]:
from rank_bm25 import BM25Okapi

STOPWORD = set("""yang dan di ke dari untuk pada dengan dalam adalah atau sebagaimana dimaksud
ayat pasal huruf angka bab ini itu oleh dapat tidak akan telah harus wajib sebagai atas terhadap
dilakukan berdasarkan sesuai antara serta bahwa maka setiap paling""".split())

def tokenisasi(t: str):
    return [w for w in re.findall(r'[a-z0-9]+', str(t).lower()) if w not in STOPWORD and len(w) > 2]

bm25 = BM25Okapi([tokenisasi(t) for t in korpus['teks_embed']])
print("BM25 siap:", len(korpus), "dokumen")

def retrieve_bm25(query: str, k: int = 10) -> pd.DataFrame:
    skor = bm25.get_scores(tokenisasi(query))
    top = np.argsort(skor)[::-1][:k]
    hasil = korpus.iloc[top].copy(); hasil['skor'] = skor[top]
    return hasil[['chunk_id','nama_dokumen','pasal','ayat','tipe','isi','skor']]

BM25 siap: 21073 dokumen


In [ ]:
def saring_metadata(df: pd.DataFrame, filter_meta: dict | None) -> pd.DataFrame:
    """filter_meta contoh: {'jenis':'UU', 'tahun':(2020,2026), 'tipe':['Sanksi','Larangan'],
                            'entitas':'Pemerintah', 'kata_kunci':'frekuensi', 'pasal':'34', 'bab':'BAB IV'}"""
    if not filter_meta: return df
    m = pd.Series(True, index=df.index)
    for kol, nilai in filter_meta.items():
        if kol == 'tahun' and isinstance(nilai, tuple):
            m &= df['tahun'].between(*nilai)
        elif kol in ('entitas', 'kata_kunci'):
            m &= df[kol].astype(str).str.contains(str(nilai), case=False, na=False)
        elif isinstance(nilai, (list, set, tuple)):
            m &= df[kol].isin(list(nilai))
        else:
            m &= df[kol].astype(str).str.lower() == str(nilai).lower()
    return df[m]

def retrieve_hybrid(query: str, k: int = 10, filter_meta: dict | None = None,
                    k_kandidat: int = 50, rrf_k: int = 60) -> pd.DataFrame:
    sub = saring_metadata(korpus, filter_meta)
    if len(sub) == 0: return sub
    # Dense — cari lebih banyak kandidat, lalu saring ke subset metadata
    qv = embedder.encode([query], normalize_embeddings=True).astype('float32')
    _, idx = faiss_index.search(qv, min(k_kandidat * 4, len(korpus)))
    rank_dense = [i for i in idx[0] if i in set(sub.index)][:k_kandidat]
    # Sparse
    skor_bm = bm25.get_scores(tokenisasi(query))
    rank_bm = [i for i in np.argsort(skor_bm)[::-1] if i in set(sub.index)][:k_kandidat]
    # RRF
    rrf = {}
    for peringkat, i in enumerate(rank_dense): rrf[i] = rrf.get(i, 0) + 1/(rrf_k + peringkat + 1)
    for peringkat, i in enumerate(rank_bm):    rrf[i] = rrf.get(i, 0) + 1/(rrf_k + peringkat + 1)
    top = sorted(rrf, key=rrf.get, reverse=True)[:k]
    hasil = korpus.loc[top].copy(); hasil['skor'] = [rrf[i] for i in top]
    return hasil[['chunk_id','nama_dokumen','pasal','ayat','tipe','isi','skor']]

# Contoh: hanya cari norma Sanksi pada regulasi tahun ≥ 2020
retrieve_hybrid("denda administratif penggunaan spektrum frekuensi radio",
                k=5, filter_meta={'tipe': ['Sanksi'], 'tahun': (2020, 2026)})

,chunk_id,nama_dokumen,pasal,ayat,tipe,isi,skor
3662,PP_46_2021__P54__a1__hb,PP Nomor 46 Tahun 2021,54,1,Sanksi,denda administratif; dan/atau,0.032787
3667,PP_46_2021__P54__a6__1,PP Nomor 46 Tahun 2021,54,6,Sanksi,Denda administratif sebagaimana dimaksud pada ayat,0.032258
3666,PP_46_2021__P54__a6__0,PP Nomor 46 Tahun 2021,54,6,Sanksi,Denda,0.031498
2639,PP_28_2025__P493__a1,PP Nomor 28 Tahun 2025,493,1,Sanksi,Setiap pemegang lzin pita frekuensi radio yang berdasarkan hasil Pengawasan ditemukan melakukan pengalihan hak pengg...,0.029857
2635,PP_28_2025__P492__a1,PP Nomor 28 Tahun 2025,492,1,Sanksi,Setiap pemegang lzin pita frekuensi radio yang berdasarkan hasil Pengawasan ditemukan melakukan kerja sama penggunaa...,0.028612


## S4 — Tahap 4: Knowledge Graph
Graf berarah dengan simpul: **dokumen → bab → pasal → satuan (chunk)**, plus simpul **entitas**.
Sisi rujukan silang diambil dari hasil ekstraksi Tahap 1 (`ref_internal`, `ref_eksternal`).

In [ ]:
KG = nx.MultiDiGraph()

def id_bab(doc, bab):     return f"{doc}::{bab}"
def id_pasal(doc, pasal): return f"{doc}::P{pasal}"

# 1) Hirarki struktural
for _, r in korpus.iterrows():
    d, b, p = r['doc_id'], id_bab(r['doc_id'], r['bab']), id_pasal(r['doc_id'], r['pasal'])
    KG.add_node(d, tipe='dokumen', label=r['nama_dokumen'], jenis=r['jenis'], tahun=int(r['tahun']) if pd.notna(r['tahun']) else None)
    KG.add_node(b, tipe='bab', label=str(r['nama_bab']))
    KG.add_node(p, tipe='pasal', label=f"Pasal {r['pasal']}")
    KG.add_node(r['chunk_id'], tipe='satuan', norma=r['tipe'], pasal=str(r['pasal']))
    KG.add_edge(d, b, rel='memuat'); KG.add_edge(b, p, rel='memuat')
    KG.add_edge(p, r['chunk_id'], rel='memuat')

# 2) Rujukan internal antar-pasal (hanya yang valid)
for _, r in ref_internal[ref_internal.valid].iterrows():
    KG.add_edge(id_pasal(r['doc_id'], r['pasal_asal']),
                id_pasal(r['doc_id'], r['pasal_tuju']), rel='merujuk')

# 3) Rujukan antar-dokumen (yang tersedia dalam korpus)
for _, r in ref_eksternal[ref_eksternal.dalam_korpus].drop_duplicates(['doc_id','doc_tuju']).iterrows():
    if r['doc_id'] != r['doc_tuju']:
        KG.add_edge(r['doc_id'], r['doc_tuju'], rel='merujuk_dokumen')

# 4) Entitas
for _, r in korpus[korpus['entitas'].notna()].iterrows():
    for ent in str(r['entitas']).split(','):
        ent = ent.strip()
        if ent:
            eid = f"ENT::{ent}"
            KG.add_node(eid, tipe='entitas', label=ent)
            KG.add_edge(r['chunk_id'], eid, rel='menyebut')

print(f"Simpul: {KG.number_of_nodes():,} | Sisi: {KG.number_of_edges():,}")
print(pd.Series([d['tipe'] for _, d in KG.nodes(data=True)]).value_counts().to_string())
print()
print(pd.Series([d['rel'] for _, _, d in KG.edges(data=True)]).value_counts().to_string())

with open(os.path.join(IFACT_DIR, 'knowledge_graph.gpickle'), 'wb') as f:
    pickle.dump(KG, f)

Simpul: 23,250 | Sisi: 78,941
satuan     21073
pasal       1984
bab          140
entitas       36
dokumen       17

memuat             63219
menyebut            9739
merujuk             5970
merujuk_dokumen       13


In [ ]:
# Statistik graf untuk laporan tesis + visualisasi sampel
deg = pd.Series({n: KG.degree(n) for n, d in KG.nodes(data=True) if d['tipe'] == 'pasal'})
print("Pasal paling terhubung (hub regulasi):")
for n, v in deg.sort_values(ascending=False).head(8).items():
    print(f"  {n}  (derajat {v})")

# Visualisasi interaktif subgraf sekitar satu pasal (buka file HTML hasilnya)
from pyvis.network import Network
def gambar_subgraf(node_pusat: str, radius: int = 1, out='subgraf.html'):
    simpul = {node_pusat}
    for _ in range(radius):
        tetangga = set()
        for s in simpul:
            tetangga |= set(KG.successors(s)) | set(KG.predecessors(s))
        simpul |= tetangga
    sub = KG.subgraph(simpul)
    warna = {'dokumen':'#d62728','bab':'#ff7f0e','pasal':'#1f77b4','satuan':'#2ca02c','entitas':'#9467bd'}
    net = Network(height='600px', directed=True, notebook=False, cdn_resources='remote')
    net.set_options('{"physics": {"enabled": false}}') # Menonaktifkan fisika melalui set_options dengan format JSON yang benar
    for n, d in sub.nodes(data=True):
        net.add_node(n, label=d.get('label', n)[:40], color=warna[d['tipe']], title=d['tipe'])
    for u, v, d in sub.edges(data=True):
        net.add_edge(u, v, label=d['rel'])
    net.save_graph(out); print("Tersimpan:", out)

gambar_subgraf(deg.idxmax(), radius=1)

Pasal paling terhubung (hub regulasi):
  UU_6_2023::P1  (derajat 1150)
  Permen_8_2026::P9  (derajat 669)
  UU_6_2023::P15  (derajat 380)
  UU_6_2023::P26  (derajat 372)
  UU_6_2023::P5  (derajat 351)
  UU_6_2023::P7  (derajat 338)
  UU_6_2023::P40  (derajat 295)
  UU_6_2023::P28  (derajat 289)
Tersimpan: subgraf.html


### Visualisasi Rujukan Antar-Dokumen (Pola Eksternal)

Fungsi di bawah ini akan membuat graf interaktif yang menunjukkan bagaimana dokumen-dokumen di dalam korpus saling merujuk berdasarkan `POLA_EKSTERNAL`. Ini hanya akan menampilkan rujukan di mana dokumen tujuan juga ada di dalam korpus.

In [ ]:
from pyvis.network import Network

def visualize_eksternal_references(ref_eksternal_df: pd.DataFrame, korpus_df: pd.DataFrame, out_file: str = 'rujukan_eksternal_interaktif.html'):
    """
    Visualizes external document-to-document references using pyvis.
    Only includes references where the target document is within the corpus.
    """
    # Filter for references within the corpus and unique (source, target) pairs
    df_in_corpus = ref_eksternal_df[ref_eksternal_df['dalam_korpus']].drop_duplicates(subset=['doc_id', 'doc_tuju'])

    if df_in_corpus.empty:
        print("Tidak ada rujukan eksternal yang menuju dokumen dalam korpus untuk divisualisasikan.")
        return

    net = Network(height='750px', width='100%', directed=True, notebook=True, cdn_resources='remote',
                  heading='Visualisasi Rujukan Antar-Dokumen (Pola Eksternal)')

    # Get unique document IDs involved in the references
    all_doc_ids = pd.concat([df_in_corpus['doc_id'], df_in_corpus['doc_tuju']]).unique()

    # Create a mapping from doc_id to nama_dokumen for node labels
    doc_labels_map = korpus_df.set_index('doc_id')['nama_dokumen'].drop_duplicates().to_dict()

    # Add nodes
    for doc_id in all_doc_ids:
        label = doc_labels_map.get(doc_id, doc_id) # Fallback to doc_id if nama_dokumen not found
        net.add_node(doc_id, label=label, title=doc_id, color='#1f77b4', size=15) # Blue nodes for documents

    # Add edges
    for _, row in df_in_corpus.iterrows():
        net.add_edge(row['doc_id'], row['doc_tuju'],
                     title=f"Merujuk dari {doc_labels_map.get(row['doc_id'], row['doc_id'])} ke {doc_labels_map.get(row['doc_tuju'], row['doc_tuju'])}",
                     arrows='to',
                     color='#d62728', # Red for reference edges
                     label='merujuk_dokumen' # Label for the edge itself
                    )

    # Save and display the graph
    net.show(out_file)
    print(f"Visualisasi rujukan eksternal interaktif tersimpan di: {out_file}")

In [ ]:
import os
from IPython.display import HTML, display

# Panggil fungsi untuk memvisualisasikan rujukan eksternal di dalam korpus
output_html_file = 'rujukan_eksternal_interaktif.html'
visualize_eksternal_references(ref_eksternal, korpus, out_file=output_html_file)

# Tampilkan visualisasi langsung di Colab
if os.path.exists(output_html_file):
    print(f"Menampilkan {output_html_file} di bawah:")
    display(HTML(filename=output_html_file))
else:
    print(f"File {output_html_file} tidak ditemukan atau gagal dibuat.")

rujukan_eksternal_interaktif.html
Visualisasi rujukan eksternal interaktif tersimpan di: rujukan_eksternal_interaktif.html
Menampilkan rujukan_eksternal_interaktif.html di bawah:


### Visualisasi Rujukan Antar-Pasal (Pola Internal)

Fungsi di bawah ini akan membuat graf interaktif yang menunjukkan bagaimana pasal-pasal di dalam korpus saling merujuk berdasarkan `POLA_INTERNAL`.

In [ ]:
from pyvis.network import Network

def visualize_internal_references(ref_internal_df: pd.DataFrame, korpus_df: pd.DataFrame, out_file: str = 'rujukan_internal_interaktif.html'):
    """
    Visualizes internal article-to-article references within documents using pyvis.
    Only includes references where the target article is valid.
    """
    # Filter for valid internal references
    df_valid_refs = ref_internal_df[ref_internal_df['valid']].copy()

    if df_valid_refs.empty:
        print("Tidak ada rujukan internal yang valid untuk divisualisasikan.")
        return

    net = Network(height='750px', width='100%', directed=True, notebook=True, cdn_resources='remote',
                  heading='Visualisasi Rujukan Antar-Pasal (Pola Internal)')
    net.set_options('{"physics": {"enabled": false}}')

    # Helper to create node IDs consistent with KG (doc_id::P{pasal})
    def get_pasal_node_id(doc_id, pasal):
        return f"{doc_id}::P{pasal}"

    # Create a mapping from doc_id to nama_dokumen for node labels
    doc_labels_map = korpus_df.set_index('doc_id')['nama_dokumen'].drop_duplicates().to_dict()

    # Add nodes and edges
    unique_doc_pasal_pairs = set()
    for _, row in df_valid_refs.iterrows():
        doc_id = row['doc_id']
        pasal_asal = str(row['pasal_asal'])
        pasal_tuju = str(row['pasal_tuju'])

        # Add document node if not already added
        if doc_id not in net.get_nodes():
            net.add_node(doc_id, label=doc_labels_map.get(doc_id, doc_id), title=doc_id, color='#d62728', size=20, font={'size':16}) # Red for document nodes

        # Add 'pasal asal' node and connect to document
        pasal_asal_node_id = get_pasal_node_id(doc_id, pasal_asal)
        if (doc_id, pasal_asal) not in unique_doc_pasal_pairs:
            net.add_node(pasal_asal_node_id, label=f"Pasal {pasal_asal}", title=f"{doc_id} - Pasal {pasal_asal}", color='#1f77b4', size=10) # Blue for pasal nodes
            net.add_edge(doc_id, pasal_asal_node_id, title='memuat', arrows='to', color='#cccccc') # Light grey for structural edges
            unique_doc_pasal_pairs.add((doc_id, pasal_asal))

        # Add 'pasal tujuan' node and connect to document
        pasal_tuju_node_id = get_pasal_node_id(doc_id, pasal_tuju)
        if (doc_id, pasal_tuju) not in unique_doc_pasal_pairs:
            net.add_node(pasal_tuju_node_id, label=f"Pasal {pasal_tuju}", title=f"{doc_id} - Pasal {pasal_tuju}", color='#1f77b4', size=10)
            net.add_edge(doc_id, pasal_tuju_node_id, title='memuat', arrows='to', color='#cccccc')
            unique_doc_pasal_pairs.add((doc_id, pasal_tuju))

        # Add internal reference edge
        net.add_edge(pasal_asal_node_id, pasal_tuju_node_id,
                     title=f"Merujuk dari Pasal {pasal_asal} ke Pasal {pasal_tuju}",
                     arrows='to',
                     color='#2ca02c', # Green for internal reference edges
                     label='merujuk')

    # Save and display the graph
    net.show(out_file)
    print(f"Visualisasi rujukan internal interaktif tersimpan di: {out_file}")

In [ ]:
import os
from IPython.display import HTML, display

# Panggil fungsi untuk memvisualisasikan rujukan internal
output_html_file_internal = 'rujukan_internal_interaktif.html'
visualize_internal_references(ref_internal, korpus, out_file=output_html_file_internal)

# Tampilkan visualisasi langsung di Colab
if os.path.exists(output_html_file_internal):
    print(f"Menampilkan {output_html_file_internal} di bawah:")
    display(HTML(filename=output_html_file_internal))
else:
    print(f"File {output_html_file_internal} tidak ditemukan atau gagal dibuat.")

rujukan_internal_interaktif.html
Visualisasi rujukan internal interaktif tersimpan di: rujukan_internal_interaktif.html
Menampilkan rujukan_internal_interaktif.html di bawah:


## S5 — Tahap 5: Graph RAG
Alur: **hybrid retrieval → seed chunks → traversal graf** untuk memperluas konteks dengan:
1. saudara satu pasal (koherensi struktural),
2. pasal yang dirujuk / merujuk (penalaran silang),
3. satuan lain yang berbagi entitas.

Skor ekspansi diberi peluruhan (*decay*) agar konteks hasil traversal tidak menenggelamkan hasil semantik.

In [ ]:
PETA_CHUNK = korpus.set_index('chunk_id')

def perluas_graf(seed_ids: list[str], bobot_seed: dict,
                 maks_per_seed: int = 4, decay: float = 0.5) -> dict:
    skor = dict(bobot_seed)
    for cid in seed_ids:
        if cid not in KG: continue
        kandidat = []
        # (a) saudara satu pasal
        for p in KG.predecessors(cid):
            if KG.nodes[p]['tipe'] == 'pasal':
                kandidat += [(c, 1.0) for c in KG.successors(p)
                             if KG.nodes[c]['tipe'] == 'satuan' and c != cid]
                # (b) pasal dirujuk / merujuk
                pasal_terkait = ([v for _, v, d in KG.out_edges(p, data=True) if d['rel']=='merujuk'] +
                                 [u for u, _, d in KG.in_edges(p, data=True)  if d['rel']=='merujuk'])
                for pt in pasal_terkait:
                    kandidat += [(c, 0.8) for c in KG.successors(pt)
                                 if KG.nodes[c]['tipe'] == 'satuan']
        # (c) berbagi entitas
        for e in KG.successors(cid):
            if KG.nodes[e]['tipe'] == 'entitas':
                kandidat += [(c, 0.5) for c in KG.predecessors(e)
                             if KG.nodes[c]['tipe'] == 'satuan' and c != cid]
        # ambil terbaik per seed
        terbaik = {}
        for c, w in kandidat: terbaik[c] = max(terbaik.get(c, 0), w)
        for c, w in sorted(terbaik.items(), key=lambda x: -x[1])[:maks_per_seed]:
            tambahan = bobot_seed[cid] * decay * w
            skor[c] = max(skor.get(c, 0), tambahan)
    return skor

def retrieve_graphrag(query: str, k: int = 10, k_seed: int = 6,
                      filter_meta: dict | None = None) -> pd.DataFrame:
    seed = retrieve_hybrid(query, k=k_seed, filter_meta=filter_meta)
    if len(seed) == 0: return seed
    bobot = dict(zip(seed['chunk_id'], seed['skor'] / seed['skor'].max()))
    skor = perluas_graf(seed['chunk_id'].tolist(), bobot)
    urut = sorted(skor, key=skor.get, reverse=True)[:k]
    hasil = PETA_CHUNK.loc[urut].reset_index()
    hasil['skor'] = [skor[c] for c in urut]
    hasil['asal'] = ['seed' if c in set(seed['chunk_id']) else 'graf' for c in urut]
    return hasil[['chunk_id','nama_dokumen','pasal','ayat','tipe','isi','skor','asal']]

retrieve_graphrag("sanksi administratif bagi penyelenggara yang tidak memenuhi kewajiban perizinan", k=8)

,chunk_id,nama_dokumen,pasal,ayat,tipe,isi,skor,asal
0,PP_28_2025__P531__a8,PP Nomor 28 Tahun 2025,531,8,Sanksi,Penyelenggara sertifikasi elektronik dikenai sanksi administratif berupa penghentian sementara kegiatan penyelenggar...,1.000000,seed
1,UU_38_2009__P40__0,UU Nomor 38 Tahun 2009,40,<NA>,Sanksi,Penyelenggara Pos yang dengan sengaja dan tanpa hak tidak menjaga keamanan dan keselamatan kiriman sebagaimana dimak...,0.994960,seed
2,UU_6_2023__P168__0,UU Nomor 6 Tahun 2023,168,<NA>,Sanksi,Penyelenggara Sarana Perkeretaapian yang tidak mengasuransikan tanggung jawabnya sebagaimana dimaksud dalam Pasal 16...,0.886777,seed
3,PP_28_2025__P531__a9,PP Nomor 28 Tahun 2025,531,9,Sanksi,Penyelenggara sertifikasi elektronik dikenai sanksi administratif berupa pemutusan akses sebagaimana dimaksud pada a...,0.875972,seed
4,UU_6_2023__P65__a3__3,UU Nomor 6 Tahun 2023,65,3,Sanksi,Setiap Pelaku Usaha yang menyelenggarakan pameran dagang dan peserta pameran dagang yang tidak memenuhi Perizinan Be...,0.869865,seed
5,UU_6_2023__P28__6,UU Nomor 6 Tahun 2023,28,<NA>,Sanksi,Penyelenggara Sarana Perkeretaapian yang mengoperasikan Sarana Perkeretaapian tidak memenuhi standar kelaikan operas...,0.853608,seed
6,PP_28_2025__P531__a1,PP Nomor 28 Tahun 2025,531,1,Sanksi,Setiap penyelenggara sertifikasi elektronik yang berdasarkan hasil Pengawasan ditemukan ketidaksesuaian atau pelangg...,0.500000,graf
7,PP_28_2025__P531__a1__hd,PP Nomor 28 Tahun 2025,531,1,Ketentuan,dikeluarkan dari daftar penyelenggara sertifikasi elektronik yang mendapat pengakuan dari menteri yang menyelenggara...,0.500000,graf


In [ ]:
# Pipeline lengkap Graph-RAG dengan LLM
res = jawab("Bagaimana ketentuan penggunaan spektrum frekuensi radio tanpa izin dan sanksinya?",
            retrieve_graphrag, k=8)
print(res['jawaban'])
print(f"\n[retrieval {res['waktu_retrieval']:.2f}s | LLM {res['waktu_llm']:.2f}s]")
res['konteks']

Berdasarkan ketentuan yang terdapat dalam Peraturan Pemerintah (PP) Nomor 46 Tahun 2021 dan Peraturan Menteri (Permen) Nomor 9 Tahun 2023, penggunaan spektrum frekuensi radio tanpa izin merupakan pelanggaran. 

Penggunaan spektrum frekuensi radio tanpa izin diatur dalam [PP Nomor 46 Tahun 2021 — Pasal 64 ayat (1)] yang mengalihkan izin penggunaan spektrum frekuensi radio tanpa persetujuan Menteri. Selain itu, [Permen Nomor 9 Tahun 2023 — Pasal 63 ayat (2)] juga menyebutkan bahwa pelanggaran pemenuhan kewajiban penggunaan spektrum frekuensi radio berupa penggunaan spektrum frekuensi radio tanpa perizinan berusaha dan/atau persetujuan dari Menteri.

Sanksi untuk penggunaan spektrum frekuensi radio tanpa izin tidak secara eksplisit diatur dalam konteks yang diberikan. Namun, dapat disimpulkan bahwa pelanggaran tersebut dapat berakibat pada pencabutan izin penggunaan spektrum frekuensi radio, sebagaimana diatur dalam [PP Nomor 46 Tahun 2021 — Pasal 64 ayat (1)]. 

Dengan demikian, ketentua

,chunk_id,nama_dokumen,pasal,ayat,tipe,isi,skor,asal
0,PP_46_2021__P64__a1__he,PP Nomor 46 Tahun 2021,64,1,Ketentuan,mengalihkan izin penggunaan Spektrum Frekuensi Radio tanpa persetujuan Menteri;,1.000000,seed
1,PP_46_2021__P63__a1__hb,PP Nomor 46 Tahun 2021,63,1,Ketentuan,permohonan penghentian izin penggunaan -Spektrum Frekuensi Radio oleh pemegang izin penggunaan Spektrum Frekuensi Ra...,0.875642,seed
2,Permen_9_2023__P3__a2,Permen Nomor 9 Tahun 2023,3,2,Kewajiban,Kewajiban BHP Spektrum Frekuensi Radio sebagaimana dimaksud pada ayat (1) mulai dikenakan pada saat izin penggunaan ...,0.846090,seed
3,Permen_9_2023__P3__a1,Permen Nomor 9 Tahun 2023,3,1,Kewajiban,Pemegang izin penggunaan Spektrum Frekuensi Radio wajib membayar BHP Spektrum Frekuensi Radio.,0.834592,seed
4,Permen_9_2023__P3__a3,Permen Nomor 9 Tahun 2023,3,3,Kewajiban,"Selain menjadi kewajiban bagi pemegang izin penggunaan Spektrum Frekuensi Radio sebagaimana dimaksud pada ayat (2), ...",0.805770,seed
5,Permen_9_2023__P63__a2,Permen Nomor 9 Tahun 2023,63,2,Kewajiban,Pelanggaran pemenuhan kewajiban penggunaan Spektrum Frekuensi Radio berdasarkan ISR berupa penggunaan Spektrum Freku...,0.734422,seed
6,PP_46_2021__P64__a1,PP Nomor 46 Tahun 2021,64,1,Ketentuan,Pengakhiran masa laku izin penggunaan Spektrum Frekuensi Radio atas dasar pencabutan sebagaimana dimaksud dalam Pasa...,0.500000,graf
7,PP_46_2021__P64__a1__ha,PP Nomor 46 Tahun 2021,64,1,Ketentuan,ain Penyelenggaraan Telekomunikasi atau IPP telah berakhir atau dicabuU,0.500000,graf
